# Query VLA Archive 
using the python API

In [1]:
import pandas as pd
import numpy as np
import pyvo
from astropy.coordinates import SkyCoord

In [2]:
meta = pd.read_csv('../ecle-meta-data.csv')
meta

,name,ra,ra_unit,dec,dec_unit,coord_ref,redshift,redshift_ref,classification,classification_ref,radio_data,radio_data_comment,otter,otter_comments
0,2019qiz,04:46:37.880,hour,-10:13:34.90,degree,2019TNSTR1857....1F,0.015100,2019TNSTR1857....1F,TDE,2019TNSCR1921....1S,True,"Kate (very good radio dataset, still need to a...",True,good uvoir dataset
1,SDSS_J0748,07:48:20.6668,hour,+47:12:14.2648,degree,2024arXiv240216951C,0.061600,2024arXiv240216951C,TDE,2024MNRAS.528.7076C,True,"Kate, VLASS",True,some xray data is available
2,SDSS_J0807,08:07:27.3157,hour,+14:05:37.0892,degree,2024arXiv240216951C,0.073800,2024arXiv240216951C,AGN,2024arXiv240216951C,True,VLASS,False,NaN
3,SDSS_J0938,09:38:01.6376,hour,+13:53:17.0423,degree,2024arXiv240216951C,0.101000,2024arXiv240216951C,AGN,2024MNRAS.528.7076C,True,"Kate, VLASS",False,NaN
4,SDSS_J0952,09:52:09.5629,hour,+21:43:13.2979,degree,2024arXiv240216951C,0.079500,2024arXiv240216951C,TDE,2024MNRAS.528.7076C,True,"Kate, VLASS",True,some xray data is available
5,SDSS_J1055,10:55:26.4177,hour,+56:37:13.1010,degree,2024arXiv240216951C,0.074000,2024arXiv240216951C,AGN,2024MNRAS.528.7076C,True,"Kate, VLASS",False,NaN
6,SDSS_J1207,12:07:19.8102,hour,+24:11:55.8789,degree,2024arXiv240216951C,0.050300,2024arXiv240216951C,AGN,2024arXiv240216951C,True,VLASS,False,NaN
7,SDSS_J1241,12:41:34.2561,hour,+44:26:39.2636,degree,2024arXiv240216951C,0.041900,2024arXiv240216951C,TDE,2024MNRAS.528.7076C,True,"Kate, VLASS, 4 other observations from other p...",False,NaN
8,SDSS_J1247,12:47:26.3719,hour,+07:05:25.0809,degree,2024arXiv240216951C,0.104000,2024arXiv240216951C,AGN,2024arXiv240216951C,True,VLASS,False,NaN
9,SDSS_J1342,13:42:44.4150,hour,+05:30:56.1451,degree,2024arXiv240216951C,0.036500,2024arXiv240216951C,TDE,2024MNRAS.528.7076C,True,"Kate, VLASS",True,minimal uvoir


## Access NRAO TAP server

In [3]:
#access NRAO TAP server
service = pyvo.dal.TAPService("https://data-query.nrao.edu/tap") 

In [5]:
# query the service
search_radius_arcseconds = 5 # in arcseconds
search_radius_deg = search_radius_arcseconds / 3600

for i, row in meta.iterrows():

    coord = SkyCoord(row.ra, row.dec, unit=(row.ra_unit,row.dec_unit))
    query_coords=coord.to_string('decimal',precision=8)
    ra_query=query_coords.split(' ')[0]
    dec_query=query_coords.split(' ')[1]

    print('\n\n##################################################################\n')
    print(row['name'], ra_query, dec_query)
    
    query = f"""
    SELECT *
    FROM tap_schema.obscore
    WHERE CONTAINS(POINT('ICRS',s_ra,s_dec),CIRCLE('ICRS',{ra_query},{dec_query},{search_radius_deg}))=1
    """
    
    res = service.search(query)
    out = res.to_table()
    
    display_keys = ['target_name', 't_min', 'center_frequencies','instrument_name',
                    'dataproduct_type', 'access_estsize', 'obs_publisher_did'] #, 'access_url',]
    out[display_keys].pprint() #max_lines=-1,max_width=-1
    #print(out.keys())

    #break

    # compute the program ID from the obs_published_did
    prog = np.unique([id.split('.')[0] for id in out['obs_publisher_did']])
    print('Program IDs:', prog)



##################################################################

2019qiz 71.65783333 -10.22636111
target_name ...
----------- ...
       TDE2 ...
       TDE2 ...
       TDE2 ...
       TDE2 ...
       TDE2 ...
       TDE2 ...
       TDE2 ...
       TDE2 ...
       TDE2 ...
       TDE2 ...
        ... ...
  AT2019QIZ ...
  AT2019QIZ ...
  AT2019QIZ ...
  AT2019QIZ ...
  AT2019QIZ ...
  AT2019QIZ ...
  AT2019QIZ ...
  AT2019QIZ ...
  AT2019qiz ...
  AT2019qiz ...
Length = 889 rows
Program IDs: ['19A-013' '20A-372' '21A-303' '24A-322'
 'VLBA_BY153_by153recor_BIN0_SRC0_0_200210T205921'
 'VLBA_BY154A_by154a_BIN0_SRC0_0_200824T205647'
 'VLBA_BY154A_by154a_BIN0_SRC0_1_200824T205647'
 'VLBA_BY154B_by154b_BIN0_SRC0_0_201215T211631'
 'VLBA_BY154B_by154b_BIN0_SRC0_1_201215T211631' 'uid___A002_Xe16acd_X406f'
 'uid___A002_Xe26f92_X13c2' 'uid___A002_Xe27761_X1545'
 'uid___A002_Xe2ada9_X401f' 'uid___A002_Xe2ada9_Xd6a2'
 'uid___A002_Xe37224_X1255b' 'uid___A002_Xe37224_X13118'
 'uid___A002_Xe37224